# Tensorflow Lite Hands-on

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pylab as plt
from ai_edge_litert.interpreter import Interpreter

from pathlib import Path

## Download data

In [ ]:
URL = (
    "https://download.microsoft.com/download/3/E/1/"
    "3E1C3F21-ECDB-4869-8368-6DEBA77B919F/"
    "kagglecatsanddogs_5340.zip"
)

base_data_dir = "../../data/"
zip_path = tf.keras.utils.get_file(
    fname="kagglecatsanddogs_5340.zip",
    origin=URL,
    extract=True,
    cache_dir=base_data_dir
)

base_dir = Path(zip_path) / "PetImages"

print(base_dir)
print((base_dir / "Cat").exists())
print((base_dir / "Dog").exists())

## Clean Bad Data

In [ ]:
from pathlib import Path
import shutil
import tensorflow as tf
from PIL import Image, UnidentifiedImageError

def clean_images(image_root, quarantine=True):
    image_root = Path(image_root)
    bad_dir = image_root.parent / "_bad_images"

    if quarantine:
        bad_dir.mkdir(exist_ok=True)

    valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}
    bad_files = []

    for image_path in image_root.rglob("*"):
        if image_path.suffix.lower() not in valid_extensions:
            continue
        try:
            if image_path.stat().st_size == 0:
                raise ValueError("empty file")
            # First: can PIL open it?
            with Image.open(image_path) as img:
                img.verify()
            # Second: can TensorFlow decode it?
            contents = tf.io.read_file(str(image_path))
            image = tf.io.decode_image(
                contents,
                channels=0,
                expand_animations=False
            )
            shape = image.shape
            if len(shape) != 3:
                raise ValueError(f"bad rank: {shape}")
            channels = shape[-1]
            if channels not in [1, 3, 4]:
                raise ValueError(f"bad channel count: {channels}")
        except Exception as e:
            bad_files.append((image_path, str(e)))
    print(f"Found {len(bad_files)} bad image files.")

    for image_path, reason in bad_files:
        print("Bad:", image_path, "|", reason)
        if quarantine:
            target = bad_dir / image_path.relative_to(image_root)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(image_path), str(target))
        else:
            image_path.unlink()

    print("Done cleaning images.")

clean_images(base_dir)

## Load data from directory

In [ ]:
IMG_SIZE = 224 # this is for mobile models, for desktop models you can use 224 or 299
BATCH_SIZE = 32
SEED = 42

raw_data = tf.keras.utils.image_dataset_from_directory(
    base_dir,
    labels="inferred",
    label_mode="int",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)

## Split data into three portions

- 80% for training
- 10% for validation
- 10% for testing

In [ ]:
num_batches = tf.data.experimental.cardinality(raw_data).numpy()

train_batches = int(0.80 * num_batches)
validation_batches = int(0.10 * num_batches)

raw_train = raw_data.take(train_batches)

remaining = raw_data.skip(train_batches)
raw_validation = remaining.take(validation_batches)
raw_test = remaining.skip(validation_batches)

train_batches = raw_train.prefetch(tf.data.AUTOTUNE)
validation_batches = raw_validation.prefetch(tf.data.AUTOTUNE)
test_batches = raw_test.prefetch(tf.data.AUTOTUNE)

print("Total batches:", num_batches)
print("Train batches:", tf.data.experimental.cardinality(raw_train).numpy())
print("Validation batches:", tf.data.experimental.cardinality(raw_validation).numpy())
print("Test batches:", tf.data.experimental.cardinality(raw_test).numpy())

## Build and Train Model from a Base Model

In [ ]:
print(
    "Using Keras Applications MobileNetV2 "
    f"with input size {IMG_SIZE}"
)

num_classes = 2
IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SHAPE,
    include_top=False,
    weights="imagenet",
    pooling="avg"
)

base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=IMG_SHAPE),
    tf.keras.layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    base_model,
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

EPOCHS = 5

hist = model.fit(
    train_batches,
    epochs=EPOCHS,
    validation_data=validation_batches
)

In [ ]:
model_dir = Path("../../models")
model_dir.mkdir(parents=True, exist_ok=True)

tflite_path = model_dir / "cats_vs_dogs_mobilenetv2.tflite"

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print("Saved TFLite model to:", tflite_path)
print("TFLite model size:", len(tflite_model), "bytes")

In [ ]:
from tqdm import tqdm
# Load TFLite model and allocate tensors.
interpreter = Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_index = input_details[0]["index"]
output_index = output_details[0]["index"]

input_shape = input_details[0]["shape"]
input_dtype = input_details[0]["dtype"]

print("Input shape expected by TFLite:", input_shape)
print("Input dtype expected by TFLite:", input_dtype)
print("Output shape:", output_details[0]["shape"])

predictions = []
test_labels = []
test_imgs = []

single_test_batches = test_batches.unbatch().batch(1)
for img, label in tqdm(single_test_batches.take(100)):
    img_np = img.numpy().astype(input_dtype)
    interpreter.set_tensor(input_index, img_np)
    interpreter.invoke()
    predictions.append(interpreter.get_tensor(output_index))

    test_labels.append(label.numpy()[0])
    test_imgs.append(img_np[0].copy())


# This will tell you how many of the predictions were correct
score = 0
for item in range(0,len(predictions)):
  prediction=np.argmax(predictions[item])
  label = test_labels[item]
  if prediction==label:
    score=score+1

print("Out of 100 predictions I got " + str(score) + " correct")



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math

class_names = ["cat", "dog"]

def plot_image(ax, i, predictions_array, true_labels, imgs):
    probs = predictions_array[i]
    true_label = true_labels[i]
    img = imgs[i]

    img = np.squeeze(img)

    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)

    predicted_label = np.argmax(probs)

    color = "green" if predicted_label == true_label else "red"

    ax.imshow(img)
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel(
        "{} {:2.0f}%\n({})".format(
            class_names[predicted_label],
            100 * np.max(probs),
            class_names[true_label]
        ),
        color=color
    )

max_index = 73
images_per_row = 5

num_rows = math.ceil(max_index / images_per_row)

plt.figure(figsize=(images_per_row * 3, num_rows * 3))

for index in range(max_index):
    ax = plt.subplot(num_rows, images_per_row, index + 1)
    plot_image(ax, index, predictions, test_labels, test_imgs)

plt.tight_layout()
plt.show()

## Further Study

To learn more about post-training quantization and optimization, please check out the user guides at https://www.tensorflow.org/lite/performance/post_training_quantization